# Malayalam TTS Dataset Builder

Builds a Malayalam speech dataset from YouTube audio using Whisper transcription.

**Before you start**
- Go to **Runtime → Change runtime type → T4 GPU** (free tier is fine)
- Run cells top to bottom — each step depends on the previous one

**Pipeline**
1. Download audio from YouTube
2. Trim to the configured window
3. Transcribe with `thennal/whisper-medium-ml`
4. Segment into TTS utterances
5. Export CSVs + audio clips to Google Drive

---
## 1. Mount Google Drive
Outputs are saved to Drive so they survive session restarts.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 2. Clone repo & install dependencies

In [3]:
import os

REPO_DIR = "/content/malayalam_tts"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ARUNJITHpm/ragam.ai.git {REPO_DIR}
else:
    print("Repo already present — pulling latest")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

Cloning into '/content/malayalam_tts'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 184 (delta 41), reused 86 (delta 34), pack-reused 84 (from 1)
Receiving objects: 100% (184/184), 80.27 MiB | 22.95 MiB/s, done.
Resolving deltas: 100% (66/66), done.
Working directory: /content/malayalam_tts


In [4]:
# Install dependencies.
# - torch is skipped: Colab already has it with CUDA/GPU support.
# - ffmpeg is already available in Colab.
import subprocess, sys

with open("requirements.txt") as f:
    deps = [
        line.strip()
        for line in f
        if line.strip() and not line.startswith("#") and not line.startswith("torch")
    ]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + deps)
print("Done.")

Done.


---
## 3. Verify GPU

In [5]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Transcription will be very slow.")
    print("Fix: Runtime > Change runtime type > T4 GPU")

GPU  : Tesla T4
VRAM : 15.6 GB


---
## 4. Configuration

**Edit this cell** to set your URLs and processing options. Everything else runs automatically.

> **Trim window:** each video is sliced from `TRIM_START_SECONDS` for `TRIM_DURATION_SECONDS`.
> Default is 60 s → 150 s (90-second sample per video).  
> Set `TRIM_DURATION_SECONDS = None` to process the full video after the start offset.

In [6]:
# ── EDIT HERE ─────────────────────────────────────────────────────────────────

VIDEO_URLS = [
    "https://www.youtube.com/watch?v=xWd9MSex8p4",
    "https://www.youtube.com/watch?v=HyWaZoOOAqY",
    # Add more URLs here ...
]

# Where to save the dataset (inside Google Drive).
OUTPUT_DIR = "/content/drive/MyDrive/malayalam_tts_dataset"

# Whisper model to use for transcription.
#   "thennal/whisper-medium-ml"  — Malayalam fine-tuned (recommended, ~1.5 GB)
#   "openai/whisper-large-v3"    — general multilingual, highest accuracy (slower)
#   "openai/whisper-medium"      — faster fallback
WHISPER_MODEL = "thennal/whisper-medium-ml"

# Trim window per video (seconds).
# Set TRIM_DURATION_SECONDS = None to use full audio after the start offset.
TRIM_START_SECONDS    = 60
TRIM_DURATION_SECONDS = 90    # None = full remaining audio

# TTS utterance length limits (seconds).
MIN_UTTERANCE_SECONDS = 3.0
MAX_UTTERANCE_SECONDS = 12.0

# Dataset split ratios (must sum to 1.0).
TRAIN_RATIO = 0.9
VAL_RATIO   = 0.05
TEST_RATIO  = 0.05

# Set True to re-generate trimmed audio + transcription even if already done.
OVERWRITE = False

# ── END EDIT ──────────────────────────────────────────────────────────────────

print(f"Videos : {len(VIDEO_URLS)}")
print(f"Output : {OUTPUT_DIR}")
print(f"Model  : {WHISPER_MODEL}")
if TRIM_DURATION_SECONDS:
    end = TRIM_START_SECONDS + TRIM_DURATION_SECONDS
    print(f"Trim   : {TRIM_START_SECONDS}s → {end}s ({TRIM_DURATION_SECONDS}s per video)")
else:
    print(f"Trim   : {TRIM_START_SECONDS}s → end of audio")

Videos : 2
Output : /content/drive/MyDrive/malayalam_tts_dataset
Model  : thennal/whisper-medium-ml
Trim   : 60s → 150s (90s per video)


---
## 5. (Optional) Upload YouTube cookies

Skip this cell for normal public videos.  
For age-restricted or sign-in-required videos, export `cookies.txt` from your browser  
using the [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc) Chrome extension, then set `UPLOAD_COOKIES = True` and run.

In [7]:
UPLOAD_COOKIES = False  # set True to trigger the file upload widget

COOKIES_PATH = None

if UPLOAD_COOKIES:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        COOKIES_PATH = f"/content/{fname}"
        print(f"Cookies saved to {COOKIES_PATH}")
    else:
        print("No file uploaded — continuing without cookies.")
else:
    print("Cookies upload skipped.")

Cookies upload skipped.


---
## 6. Write config and run the pipeline

In [8]:
import os

os.chdir(REPO_DIR)

url_block = "\n".join(f'  - "{u}"' for u in VIDEO_URLS)
trim_dur_line = (
    f"  trim_duration_seconds: {TRIM_DURATION_SECONDS}"
    if TRIM_DURATION_SECONDS is not None
    else "  trim_duration_seconds: null"
)
cookie_line = (
    f'  cookie_file: "{COOKIES_PATH}"' if COOKIES_PATH else "  cookie_file: null"
)

config_yaml = f"""video_urls:
{url_block}

output_dir: "{OUTPUT_DIR}"

audio:
  sample_rate: 16000
  channels: 1
  trim_start_seconds: {TRIM_START_SECONDS}
{trim_dur_line}
  min_utterance_seconds: {MIN_UTTERANCE_SECONDS}
  max_utterance_seconds: {MAX_UTTERANCE_SECONDS}
  silence_thresh_dbfs: -40
  min_silence_len_ms: 350
  keep_silence_ms: 120

asr:
  whisper_model: "{WHISPER_MODEL}"
  language: "ml"
  keep_empty_segments: false

youtube:
{cookie_line}
  use_browser_cookies: false
  sleep_between_downloads: 10
  ydl_sleep_interval: 5
  ydl_max_sleep_interval: 15
  retries: 5
  ratelimit: 1000000
  fetch_title_for_existing_audio: false

split:
  train_ratio: {TRAIN_RATIO}
  val_ratio: {VAL_RATIO}
  test_ratio: {TEST_RATIO}
  random_seed: 42

runtime:
  overwrite_outputs: {str(OVERWRITE).lower()}
  reuse_existing_audio: true
"""

CONFIG_PATH = "/content/colab_config.yaml"
with open(CONFIG_PATH, "w") as f:
    f.write(config_yaml)

print("Config written to", CONFIG_PATH)
print()
print(config_yaml)

Config written to /content/colab_config.yaml

video_urls:
  - "https://www.youtube.com/watch?v=xWd9MSex8p4"
  - "https://www.youtube.com/watch?v=HyWaZoOOAqY"

output_dir: "/content/drive/MyDrive/malayalam_tts_dataset"

audio:
  sample_rate: 16000
  channels: 1
  trim_start_seconds: 60
  trim_duration_seconds: 90
  min_utterance_seconds: 3.0
  max_utterance_seconds: 12.0
  silence_thresh_dbfs: -40
  min_silence_len_ms: 350
  keep_silence_ms: 120

asr:
  whisper_model: "thennal/whisper-medium-ml"
  language: "ml"
  keep_empty_segments: false

youtube:
  cookie_file: null
  use_browser_cookies: false
  sleep_between_downloads: 10
  ydl_sleep_interval: 5
  ydl_max_sleep_interval: 15
  retries: 5
  ratelimit: 1000000
  fetch_title_for_existing_audio: false

split:
  train_ratio: 0.9
  val_ratio: 0.05
  test_ratio: 0.05
  random_seed: 42

runtime:
  overwrite_outputs: false
  reuse_existing_audio: true



In [9]:
import subprocess, sys

cmd = [sys.executable, "scripts/run_dataset_build.py", "--config", CONFIG_PATH]
if OVERWRITE:
    cmd.append("--overwrite")

result = subprocess.run(cmd)  # streams output live

print()
if result.returncode == 0:
    print("Pipeline completed successfully.")
elif result.returncode == 1:
    print("Pipeline finished with errors — check the output above.")
elif result.returncode == 2:
    print("Configuration error — check the config cell above.")


Pipeline finished with errors — check the output above.


---
## 7. View results

In [10]:
import json
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / "run_summary.json"

if not summary_path.exists():
    print(f"run_summary.json not found at {summary_path}")
else:
    s = json.loads(summary_path.read_text())

    print("=" * 40)
    print(f"  Videos processed : {s['total_urls']}")
    print(f"  Successful        : {s['successful_videos']}")
    print(f"  Failed            : {s['failed_videos']}")
    print(f"  Total utterances  : {s['total_utterances']}")
    print(f"  Total audio       : {s['total_duration_seconds'] / 60:.1f} min")
    print(f"  Avg utterance     : {s['average_utterance_duration']:.1f} s")
    print(f"  Train / Val / Test: {s['splits']['train_count']} / {s['splits']['val_count']} / {s['splits']['test_count']}")
    print("=" * 40)

    if s.get("errors"):
        print(f"\nErrors ({len(s['errors'])}) :")
        for e in s["errors"]:
            print(f"  [{e['stage']}] {e['video_id']}: {e['error'][:100]}")

    print(f"\nOutputs saved to: {OUTPUT_DIR}")

run_summary.json not found at /content/drive/MyDrive/malayalam_tts_dataset/run_summary.json


---
## 8. (Optional) Push run summary to GitHub

Commits `run_summary.json` (and `run_log.txt` if present) to a `colab-runs` branch under `runs/<timestamp>/` so you can pull it locally for analysis.

**One-time setup:**
1. Create a GitHub Personal Access Token with `repo` scope at https://github.com/settings/tokens
2. In Colab, click the **key icon** (left sidebar) → **Add new secret**
3. Name: `GH_TOKEN`, Value: your token, toggle **Notebook access** on

**Pull locally:**
```
git fetch origin colab-runs
git checkout colab-runs -- runs/
```

In [11]:
import datetime
import shutil
import subprocess
from pathlib import Path

GH_USER   = "ARUNJITHpm"
GH_REPO   = "ragam.ai"
GH_BRANCH = "colab-runs"

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = None

if not GH_TOKEN:
    print("GH_TOKEN not found in Colab Secrets. See the markdown cell above for setup.")
else:
    summary_src = Path(OUTPUT_DIR) / "run_summary.json"
    if not summary_src.exists():
        print(f"No run_summary.json at {summary_src} — run the pipeline first.")
    else:
        runs_repo = Path("/content/colab_runs_repo")
        remote = f"https://{GH_TOKEN}@github.com/{GH_USER}/{GH_REPO}.git"

        if runs_repo.exists():
            shutil.rmtree(runs_repo)

        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GH_BRANCH, remote, str(runs_repo)],
            capture_output=True, text=True,
        )
        if clone.returncode != 0:
            print(f"Branch '{GH_BRANCH}' not on remote — creating it as an orphan branch.")
            subprocess.check_call(["git", "clone", "--depth", "1", remote, str(runs_repo)])
            subprocess.check_call(["git", "-C", str(runs_repo), "checkout", "--orphan", GH_BRANCH])
            subprocess.check_call(["git", "-C", str(runs_repo), "rm", "-rf", "."])

        subprocess.check_call(["git", "-C", str(runs_repo), "config", "user.email", "colab@runs.local"])
        subprocess.check_call(["git", "-C", str(runs_repo), "config", "user.name",  "Colab Runner"])

        ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        dest = runs_repo / "runs" / ts
        dest.mkdir(parents=True, exist_ok=True)

        shutil.copy2(summary_src, dest / "run_summary.json")

        log_src = Path(REPO_DIR) / "run_log.txt"
        if log_src.exists():
            shutil.copy2(log_src, dest / "run_log.txt")

        subprocess.check_call(["git", "-C", str(runs_repo), "add", "."])
        subprocess.check_call(["git", "-C", str(runs_repo), "commit", "-m", f"Colab run {ts}"])
        subprocess.check_call(["git", "-C", str(runs_repo), "push", "origin", GH_BRANCH])

        print(f"\nPushed to {GH_USER}/{GH_REPO}@{GH_BRANCH}: runs/{ts}/")
        print(f"Locally: git fetch origin {GH_BRANCH} && git checkout {GH_BRANCH} -- runs/{ts}/")

GH_TOKEN not found in Colab Secrets. See the markdown cell above for setup.


---
## 9. Download deliverables to local machine

Bundles only the small outputs you need locally:

- `run_summary.json`
- All `*.csv` (dataset, metadata, train/val/test splits)
- `transcripts/` (per-video segment JSONs)
- `segments/` (final utterance WAVs)

Skips the bulky `audio_full/` and `audio_trimmed/` directories — those stay in Drive and can be re-derived. Browser triggers a download of `run_outputs.zip` — unzip into your local project folder to replace previous outputs.

In [12]:
import zipfile
from pathlib import Path
from google.colab import files as colab_files

out = Path(OUTPUT_DIR)
zip_path = "/content/run_outputs.zip"

INCLUDE_DIRS  = ("transcripts", "segments")
INCLUDE_FILES = ("run_summary.json",)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in INCLUDE_FILES:
        src = out / fname
        if src.exists():
            zf.write(src, fname)

    for csv in out.glob("*.csv"):
        zf.write(csv, csv.name)

    for sub in INCLUDE_DIRS:
        sub_path = out / sub
        if not sub_path.exists():
            continue
        for f in sub_path.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(out))

zip_size_mb = Path(zip_path).stat().st_size / 1e6
print(f"Bundled {zip_size_mb:.1f} MB — starting browser download.")
colab_files.download(zip_path)

Bundled 0.0 MB — starting browser download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 10. (Optional) Download the *full* dataset (including raw audio) as ZIP

The previous cell already downloaded the deliverables. Use this only if you also want the bulky raw + trimmed audio for archival.

In [13]:
import shutil
from google.colab import files as colab_files
from pathlib import Path

zip_base = "/content/malayalam_tts_dataset"
shutil.make_archive(zip_base, "zip", Path(OUTPUT_DIR))
colab_files.download(zip_base + ".zip")
print("Download started.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/malayalam_tts_dataset'